# MAE 6246 - Introduction to MuJoCo: The Furuta Pendulum

**Name:** _Enter your name._  
**Total:** 100 points  
**Due:** To be announced

## Submission format

Submit the completed IPYNB file through Blackboard. Enter analytical work and explanations in the designated Markdown response cells using LaTeX or clearly formatted algebra. If you are not familiar with LaTeX, you may instead write the analytical work on paper and submit it as a PDF file in addition to the IPYNB file. In that case, write **submitted separately as a PDF** in each corresponding response cell.

Before submitting, restart the kernel and run all cells from top to bottom. Keep all requested figures, numerical output, and the embedded animation in the notebook. Complete all `YOUR CODE HERE` sections; function stubs raise `NotImplementedError` until you implement them. Use SI units and radians internally.

The supplied video-display cell uses `Video(..., embed=True)`, which stores the animation in the saved notebook. **A separate MP4 submission is not required.** Save and reopen the notebook to confirm that its video output is included. Submit the notebook with its outputs intact.

## Motivation

A Furuta pendulum consists of a horizontal arm driven about a vertical axis and a passive pendulum attached to the arm's tip. You will simulate a supplied MuJoCo model, compare its state histories with an independent numerical solution of the nonlinear equations, and generate an animation.

This assignment asks three questions:

1. How do we initialize a physics simulator and apply a prescribed input?
2. Does its simulated motion agree with the supplied nonlinear equations?
3. How do we turn recorded states into an animation?

Begin with the official [MuJoCo introduction](https://mujoco.readthedocs.io/en/stable/overview.html#introduction) and [Python tutorial notebook](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb). Problem 0 guides this preparation.

The model and equations are supplied; you are not asked to derive the mechanics or design a controller or observer. We will reuse this plant in later assignments.

## Problem 0 - Official MuJoCo tutorial (ungraded)

Before working with the Furuta pendulum:

1. Read the official [MuJoCo introduction](https://mujoco.readthedocs.io/en/stable/overview.html#introduction).
2. Open the [official Python tutorial in Colab](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb), or download it from the [MuJoCo repository](https://github.com/google-deepmind/mujoco/blob/main/python/tutorial.ipynb) and run it locally. Read the explanations and follow the runnable examples.
3. Pay particular attention to loading an XML model, the roles of `MjModel` and `MjData`, accessing joint positions and velocities, advancing the simulation with `mj_step`, and rendering images and video.
4. Confirm that you can run a simulation and display its animation before starting the pendulum problems.

This preparation is ungraded. You do not need to submit a separate tutorial notebook or screenshots.

## Setup

This notebook includes the complete XML model and rendering helper, so no companion files are required. The next cell installs missing packages into the current Python environment. The model and helper were checked with **MuJoCo 3.17.0**.

Numerical simulation and plots do not require a graphics context; video rendering does. Colab/Linux without a display uses EGL. Resolve any graphics issue during the tutorial or the supplied graphics check before beginning Problem 4.

In [ ]:
import importlib.util
import subprocess
import sys
import os

packages = {
    "mujoco": "mujoco==3.17.0", "numpy": "numpy", "scipy": "scipy",
    "matplotlib": "matplotlib", "PIL": "pillow",
    "imageio_ffmpeg": "imageio-ffmpeg",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")  # Set before importing MuJoCo.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mujoco
from scipy.integrate import solve_ivp
from IPython.display import display, Video

print("MuJoCo version:", mujoco.__version__)
if mujoco.__version__ != "3.17.0":
    print("The supplied example was validated with 3.17.0; record your version in your submission.")

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})


## Physical model and coordinate conventions — supplied

The arm has length $R$ and rotates in a horizontal plane. Its angle is $\phi$, positive counterclockwise when viewed from above. When $\phi=0$, the arm extends along world $+x$. The pendulum rotates about the arm's radial axis. Its angle is $\theta$, with **$\theta=0$ upright** and $\theta=\pi$ downward. Positive $\theta$ tilts the pendulum toward the arm's local $+y$ direction. In the XML, this is implemented by a pendulum hinge axis of `-1 0 0`.

The state and input are

$$
x=\begin{bmatrix}\phi&\theta&\dot\phi&\dot\theta\end{bmatrix}^{T},
\qquad u=\text{torque applied to the arm hinge}\quad[\mathrm{N\,m}].
$$

Let $Q_z(\phi)$ denote a positive rotation about world $+z$. Relative to the arm pivot, the pendulum center of mass is

$$
r_c=Q_z(\phi)
\begin{bmatrix}R\\ \ell\sin\theta\\ \ell\cos\theta\end{bmatrix}.
$$

The arm pivot is 0.45 m above the floor. Both moving links are uniform solid cylinders; their inertias are explicitly specified in the XML. The hub and tip marker are visual decorations and add no mass or inertia. There is no damping, dry friction, actuator dynamics, torque saturation, joint limit, or contact. The floor is visual only. The motor has unit gear ratio, so its control value equals applied arm torque. The pendulum joint is unactuated.

| Symbol | Meaning | Value |
|---|---|---:|
| $R$ | Arm length | 0.20 m |
| $m_a$ | Arm mass | 0.20 kg |
| $r_a$ | Arm cylinder radius | 0.012 m |
| $m$ | Pendulum mass | 0.10 kg |
| $L$ | Full pendulum length | 0.30 m |
| $r_p$ | Pendulum cylinder radius | 0.008 m |
| $\ell=L/2$ | Pendulum pivot-to-COM distance | 0.15 m |
| $g$ | Gravitational acceleration | 9.81 m/s² |

The arm inertia about its **vertical pivot axis**, including the parallel-axis term, is
$$
J_a=\frac{m_a}{12}(R^2+3r_a^2)+m_a(R/2)^2
=0.002673866666666667\ \mathrm{kg\,m^2}.
$$
The pendulum principal inertias about its **center of mass** are
$$
J_t=\frac{m}{12}(L^2+3r_p^2)=0.0007516\ \mathrm{kg\,m^2},
\qquad J_z=\frac{m r_p^2}{2}=0.0000032\ \mathrm{kg\,m^2}.
$$
Here $J_t$ is the transverse inertia and $J_z$ is the inertia about the pendulum's long axis. Both are retained in the equations; do not set $J_z=0$.

### Nonlinear equations

Define
$$
H=m\ell^2+J_t,\qquad D=H-J_z,\qquad \beta=mR\ell,
$$
and
$$
a(\theta)=J_a+mR^2+J_z+D\sin^2\theta,
\qquad
M(\theta)=\begin{bmatrix}
a(\theta)&\beta\cos\theta\\
\beta\cos\theta&H
\end{bmatrix}.
$$
The full nonlinear dynamics of the supplied model are
$$
M(\theta)
\begin{bmatrix}\ddot\phi\\\ddot\theta\end{bmatrix}
=
\begin{bmatrix}
u-2D\sin\theta\cos\theta\,\dot\phi\dot\theta
  +\beta\sin\theta\,\dot\theta^2\\
D\sin\theta\cos\theta\,\dot\phi^2+mg\ell\sin\theta
\end{bmatrix}.
$$
Thus $\dot x=[\dot\phi,\dot\theta,\ddot\phi,\ddot\theta]^T$. Use `np.linalg.solve` to obtain the two accelerations from this $2\times2$ system. Do not replace the trigonometric functions with small-angle approximations.

For reference, the kinetic and potential energies are
$$
T=\tfrac12 a(\theta)\dot\phi^2+
\beta\cos\theta\,\dot\phi\dot\theta+
\tfrac12H\dot\theta^2,
\qquad V=mg\ell\cos\theta,
$$
where an irrelevant constant in potential energy is omitted. These expressions specify the same model; an energy check is optional.

### Supplied MuJoCo model

The following cell defines the XML and a model-loading helper. You may inspect the XML, but keep the physical parameters and assumptions unchanged for the required experiments.

In [ ]:
MODEL_XML = r'''
<mujoco model="MAE 6246 Furuta pendulum">
  <compiler angle="radian" inertiafromgeom="false"/>
  <option timestep="0.001" gravity="0 0 -9.81" integrator="RK4"/>
  <visual>
    <global offwidth="960" offheight="640"/>
    <headlight ambient="0.5 0.5 0.5" diffuse="0.7 0.7 0.7"/>
  </visual>
  <default>
    <joint damping="0" frictionloss="0" armature="0" limited="false"/>
    <geom contype="0" conaffinity="0"/>
  </default>
  <worldbody>
    <light pos="0 -1 2" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="1 1 0.01" rgba="0.91 0.93 0.96 1"/>
    <geom name="base" type="cylinder" pos="0 0 0.025" size="0.08 0.025" rgba="0.2 0.25 0.32 1"/>
    <geom name="stand" type="cylinder" pos="0 0 0.25" size="0.018 0.20" rgba="0.4 0.45 0.50 1"/>
    <!-- Fixed reference: red = world +x; green = world +y. -->
    <geom type="capsule" fromto="0 0 0.006 0.3 0 0.006" size="0.003" rgba="0.75 0.2 0.2 1"/>
    <geom type="capsule" fromto="0 0 0.006 0 0.3 0.006" size="0.003" rgba="0.2 0.6 0.3 1"/>
    <body name="arm" pos="0 0 0.45">
      <!-- phi: positive counterclockwise viewed from above. -->
      <joint name="arm_hinge" type="hinge" axis="0 0 1"/>
      <!-- Solid cylinder: mass .2, length .2, radius .012, along local x. -->
      <inertial pos="0.1 0 0" mass="0.2"
                diaginertia="0.0000144 0.000673866666666667 0.000673866666666667"/>
      <geom name="arm_shape" type="cylinder" fromto="0 0 0 0.2 0 0" size="0.012" rgba="0.10 0.40 0.65 1"/>
      <geom name="hub" type="sphere" size="0.022" rgba="0.15 0.2 0.3 1"/>
      <body name="pendulum" pos="0.2 0 0">
        <!-- theta=0: upright. Positive theta tilts toward arm-local +y. -->
        <joint name="pendulum_hinge" type="hinge" axis="-1 0 0"/>
        <!-- Solid cylinder: mass .1, length .3, radius .008, along local z. -->
        <inertial pos="0 0 0.15" mass="0.1"
                  diaginertia="0.0007516 0.0007516 0.0000032"/>
        <geom name="pendulum_shape" type="cylinder" fromto="0 0 0 0 0 0.3" size="0.008" rgba="0.94 0.46 0.12 1"/>
        <geom name="tip_marker" type="sphere" pos="0 0 0.3" size="0.013" rgba="0.95 0.65 0.15 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <motor name="arm_torque" joint="arm_hinge" gear="1" ctrllimited="false"/>
  </actuator>
  <sensor>
    <jointpos name="arm_angle" joint="arm_hinge"/>
    <jointpos name="pendulum_angle" joint="pendulum_hinge"/>
  </sensor>
</mujoco>
'''

def make_model(h=0.001):
    model = mujoco.MjModel.from_xml_string(MODEL_XML)
    model.opt.timestep = h
    return model

# Optional: save a standalone copy for inspection in a text editor.
Path("furuta.xml").write_text(MODEL_XML)

model = make_model()
print("nq, nv, nu =", model.nq, model.nv, model.nu)


### Graphics check - supplied, ungraded

The blue link is the driven arm; the orange link is the passive pendulum. The red and green floor lines indicate world +x and +y. This cell renders the upright configuration without advancing time.

In [ ]:
preview_model = make_model()
preview_data = mujoco.MjData(preview_model)
mujoco.mj_forward(preview_model, preview_data)
camera = mujoco.MjvCamera()
camera.lookat[:] = [0, 0, 0.43]
camera.distance, camera.azimuth, camera.elevation = 1.25, 135, -20
try:
    with mujoco.Renderer(preview_model, height=480, width=720) as renderer:
        renderer.update_scene(preview_data, camera=camera)
        plt.figure(figsize=(9, 6))
        plt.imshow(renderer.render())
        plt.axis("off")
        plt.show()
except Exception as exc:
    print("Rendering is unavailable in this environment:", exc)
    print("Numerical work can continue. Resolve graphics support before Problem 4.")


## Problem 1 - Interpret and initialize the model (20 points)

### (a) Coordinates and model structure (5 points)

Explain why this system has two generalized coordinates, four states, and one input. Identify the two hinge axes and explain which joint receives torque. State which equilibrium is represented by $\theta=0$. Identify how the XML disables contact and specifies the moving-body inertias.

### Response for Problem 1(a)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

### (b) State mapping (10 points)

Complete `set_state` and `read_state` below using the joint names `arm_hinge` and `pendulum_hinge`. `set_state` should assign positions and velocities and call `mujoco.mj_forward(model, data)` to update derived quantities. Explain the distinction between the model (`MjModel`) and the simulation data (`MjData`).

In [ ]:
def set_state(model, data, x):
    """Assign x = [phi, theta, phi_dot, theta_dot] and refresh derived data."""
    # YOUR CODE HERE
    raise NotImplementedError

def read_state(data):
    """Return a new length-4 NumPy array in the assignment's state order."""
    # YOUR CODE HERE
    raise NotImplementedError

### Response for Problem 1(b)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

### (c) Initialization check (5 points)

For a fresh `MjData`, set
$$
x_{\mathrm{test}}=[0.3,-0.4,0.7,-0.5]^T.
$$
Read the state back and report the largest absolute difference. Verify that initialization does not advance `data.time`. Explain why prescribing joint positions at every simulation step would not simulate the system's response to torque.

In [ ]:
# YOUR CODE HERE: initialize x_test, read it back, and report the error and time.

### Response for Problem 1(c)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

## Problem 2 - Simulate prescribed initial conditions and torques (30 points)

Both experiments use $0\le t\le3$ s and a baseline physics timestep $h=0.001$ s. Keep the supplied RK4 integrator.

| Case | Initial state $[\phi,\theta,\dot\phi,\dot\theta]^T$ | Applied torque |
|---|---|---|
| A: free motion near upright | $[0,5\pi/180,0,0]^T$ | $u=0$ throughout |
| B: forced motion near downward | $[0,\pi-10\pi/180,0,0]^T$ | $+0.015$ N m for $0\le t<0.5$; $−0.015$ N m for $0.5\le t<1$; zero for $1\le t\le3$ |

### (a) Simulation loop (20 points)

Complete `simulate_mujoco(case, h)`. Create a fresh model and data object for each run. Initialize the state once, hold the specified torque over each interval, and advance the dynamics using `mujoco.mj_step`. Store time and **copies** of the four states, including the initial state and the state at 3 s. Return `times` with shape `(N+1,)` and `states` with shape `(N+1,4)`.

Use integer step indices to align torque switches exactly with 0.5 s and 1 s. Avoid deciding the switching interval from an accumulated floating-point time. The prescribed timestep divides the input intervals exactly. Do not reset positions or velocities inside the integration loop.

In [ ]:
CASES = {
    "A": {"x0": np.array([0., np.deg2rad(5), 0., 0.]),
          "breaks": np.array([0., 3.]), "inputs": np.array([0.])},
    "B": {"x0": np.array([0., np.pi-np.deg2rad(10), 0., 0.]),
          "breaks": np.array([0., 0.5, 1., 3.]),
          "inputs": np.array([0.015, -0.015, 0.])},
}

def simulate_mujoco(case, h=0.001):
    """Return (times, states) for the supplied piecewise-constant input."""
    # YOUR CODE HERE
    raise NotImplementedError

### Response for Problem 2(a)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

### (b) Checks and interpretation (10 points)

Run both cases. Report the number of samples and verify the initial and final times and initial state. Keep angles continuous; do not wrap them to $[-\pi,\pi]$. Describe the free motion in Case A and explain why starting exactly at rest at upright would be a poor demonstration of instability.

In [ ]:
mj_results = {}
for name, case in CASES.items():
    times, states = simulate_mujoco(case, h=0.001)
    mj_results[name] = (times, states)
    assert states.shape == (3001, 4)
    np.testing.assert_allclose(times[[0, -1]], [0., 3.], atol=1e-12)
    np.testing.assert_allclose(states[0], case["x0"], atol=1e-12)
    assert np.isfinite(states).all()
    print(name, "samples:", len(times), "final state:", states[-1])


### Response for Problem 2(b)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

## Problem 3 - Independently integrate the nonlinear equations (30 points)

### (a) Right-hand side and acceleration check (10 points)

Complete `nonlinear_rhs(t, x, u)` using only the supplied equations and parameters. This function must not use MuJoCo to compute accelerations.

At $x_{\mathrm{test}}=[0.3,-0.4,0.7,-0.5]^T$ and $u=0.012$ N m, compare the two accelerations from your function with MuJoCo's accelerations after setting the state, setting `data.ctrl[0]`, and calling `mujoco.mj_forward`. Report both acceleration vectors and their absolute differences. Agreement within $10^{-9}$ rad/s² is expected. Explain why this comparison helps separate a modeling error from a numerical integration error.

In [ ]:
# Supplied parameters, computed from the dimensions above.
R, ARM_MASS, ARM_RADIUS = 0.20, 0.20, 0.012
MASS, LENGTH, PEND_RADIUS, GRAVITY = 0.10, 0.30, 0.008, 9.81
ELL = LENGTH / 2
JA = ARM_MASS*(R**2 + 3*ARM_RADIUS**2)/12 + ARM_MASS*(R/2)**2
JT = MASS*(LENGTH**2 + 3*PEND_RADIUS**2)/12
JZ = MASS*PEND_RADIUS**2/2
H = MASS*ELL**2 + JT
D = H - JZ
BETA = MASS*R*ELL

def nonlinear_rhs(t, x, u):
    """Return [phi_dot, theta_dot, phi_ddot, theta_ddot]."""
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: compare accelerations at x_test with u=0.012 N m.

### Response for Problem 3(a)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

### (b) Independent integration (10 points)

Complete `simulate_ode(case, times)` using `solve_ivp` with method `DOP853`, relative tolerance $10^{-11}$, and absolute tolerance $10^{-13}$. Use the same initial state and torque history as MuJoCo. **Restart the ODE solver at each torque switch** and pass the final state of one interval to the next. Evaluate the solution at the requested times using dense output or `t_eval`. Check `sol.success`.

The two simulations must advance their own states independently; do not periodically reset either simulation to the other trajectory. Evaluate the nonlinear ODE at the exact MuJoCo sample times for comparison.

In [ ]:
def simulate_ode(case, times):
    """Return nonlinear ODE states with shape (len(times),4)."""
    # YOUR CODE HERE
    raise NotImplementedError

ode_results = {}
for name, case in CASES.items():
    times, _ = mj_results[name]
    ode_results[name] = simulate_ode(case, times)


### Response for Problem 3(b)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

### (c) Compare state histories (10 points)

For each case, make four labeled state-history panels with the MuJoCo and nonlinear ODE trajectories overlaid. Use the same sample times for both methods. Plot the prescribed torque separately as a step plot. Label time in seconds, angles in radians, angular velocities in rad/s, and torque in N m. Distinguish the methods with line styles or markers and include a legend.

Describe whether the trajectories agree and whether this is consistent with your acceleration check. Relate one feature of each response to the prescribed initial state or input.

In [ ]:
# YOUR CODE HERE: state overlays and torque histories for Cases A and B.

### Response for Problem 3(c)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

## Problem 4 - Generate an animation (20 points)

### (a) Recorded-trajectory animation (15 points)

Use the supplied helper to make a **3-second, 30-fps MP4 of Case A**, using your saved MuJoCo state history. Embed the result in the notebook using the supplied display cell; no separate MP4 submission is required. The helper supplies a fixed camera, time display, and angle labels. You may also animate Case B, but only Case A is required.

In [ ]:
"""Provided helper: render a recorded trajectory without simulating it again."""
from pathlib import Path
import numpy as np
import mujoco


def render_trajectory(model, times, states, filename="furuta.mp4", fps=30):
    """States use [phi, theta, phi_dot, theta_dot], in radians and rad/s.

    Angles must be continuous (not wrapped). Interpolation is only for display.
    The final endpoint is omitted so video duration equals the simulation span.
    Uses a separate MjData; never changes the recorded simulation state.
    """
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib
    import imageio_ffmpeg

    times, states = np.asarray(times), np.asarray(states)
    if states.shape != (len(times), 4) or len(times) < 2:
        raise ValueError("Expected times (N,) and states (N,4), N >= 2")
    if not np.all(np.diff(times) > 0) or not np.isfinite(states).all():
        raise ValueError("Times must increase and states must be finite")
    if fps <= 0:
        raise ValueError("fps must be positive")
    frame_count = int(round((times[-1] - times[0]) * fps))
    if frame_count < 1 or not np.isclose(frame_count / fps, times[-1] - times[0]):
        raise ValueError("Duration times fps must be a positive integer")
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    data = mujoco.MjData(model)
    camera = mujoco.MjvCamera()
    camera.lookat[:] = [0, 0, 0.43]
    camera.distance, camera.azimuth, camera.elevation = 1.25, 135, -20
    font = ImageFont.truetype(
        str(Path(matplotlib.get_data_path()) / "fonts/ttf/DejaVuSans.ttf"), 20)
    writer = imageio_ffmpeg.write_frames(
        str(filename), (960, 640), fps=fps, codec="libx264",
        pix_fmt_out="yuv420p", quality=8, macro_block_size=16,
        output_params=["-movflags", "+faststart"])
    writer.send(None)
    try:
        with mujoco.Renderer(model, height=640, width=960) as renderer:
            for k in range(frame_count):
                t = times[0] + k / fps
                x = np.array([np.interp(t, times, states[:, j]) for j in range(4)])
                data.joint("arm_hinge").qpos[0] = x[0]
                data.joint("pendulum_hinge").qpos[0] = x[1]
                data.joint("arm_hinge").qvel[0] = x[2]
                data.joint("pendulum_hinge").qvel[0] = x[3]
                data.time = t
                mujoco.mj_forward(model, data)
                renderer.update_scene(data, camera=camera)
                frame = Image.fromarray(renderer.render())
                draw = ImageDraw.Draw(frame)
                draw.rectangle((0, 0, 960, 78), fill="#f5f7fb")
                draw.text((24, 12), f"Furuta pendulum | recorded MuJoCo trajectory | t = {t:.2f} s",
                          fill="#22344a", font=font)
                draw.text((24, 43),
                          f"Arm: {np.rad2deg(x[0]):+.1f} deg   Pendulum: {np.rad2deg(x[1]):+.1f} deg (0 = upright)",
                          fill="#22344a", font=font)
                writer.send(np.asarray(frame))
    finally:
        writer.close()
    return filename


In [ ]:
times_A, states_A = mj_results["A"]
video_path = render_trajectory(make_model(), times_A, states_A,
                               filename="furuta_case_A.mp4", fps=30)
display(Video(str(video_path), embed=True))


### (b) Interpretation (5 points)

Identify one feature of the video and relate it to the corresponding state-history plot. Explain the difference between the physics timestep (1 ms) and the video frame interval (1/30 s). Explain why changing the video frame rate should not change the simulated state history.

The helper uses a separate `MjData` to replay recorded states for display. Assigning states during replay is appropriate; assigning them during the torque-driven simulation would replace the dynamics. The helper calls `mj_forward` to update geometry and never calls `mj_step`. Its interpolation is for display only and is not used in the numerical comparisons.

### Response for Problem 4(b)

Replace this text with your explanation and requested numerical results. If you submit handwritten work as a separate PDF, write **submitted separately as a PDF** here.

## Submission checklist

- [ ] I followed the official MuJoCo tutorial before starting the pendulum problems.
- [ ] I explained the model and verified the state initialization in Problem 1.
- [ ] I simulated both prescribed experiments and checked their initial states and time ranges.
- [ ] I reported the acceleration comparison and independently integrated the nonlinear equations.
- [ ] I included state-history overlays and input plots for both experiments.
- [ ] I embedded the Case A animation and related its motion to the state histories.
- [ ] I completed the written response cells, restarted the kernel, and ran the notebook from top to bottom.
- [ ] I saved the notebook with all outputs, including the embedded video.

## References

- [MuJoCo introduction](https://mujoco.readthedocs.io/en/stable/overview.html#introduction)
- [Official MuJoCo Python tutorial](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb)
- [MuJoCo Python interface](https://mujoco.readthedocs.io/en/stable/python.html)
- [MuJoCo simulation and initialization](https://mujoco.readthedocs.io/en/stable/programming/simulation.html)
- [MuJoCo XML reference](https://mujoco.readthedocs.io/en/stable/XMLreference.html)
- [SciPy `solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)
- [Cazzolato and Prime, *On the Dynamics of the Furuta Pendulum*](https://doi.org/10.1155/2011/528341) - background on retaining rotational inertia terms. Use the coordinate conventions and equations supplied here; signs and notation in other references can differ.
